# CNN in PyTorch — live coding lab

**Course:** Master's Degree in Computer Engineering  
**Format:** live coding + short individual activities  
**Reference:** `PyTorch/12_deep_computer_vision_with_cnns.ipynb` from the DeepLearning repository.

## Goals
By the end of the lab, students will be able to:
1. clearly distinguish **CPU, GPU, and data transfers** in PyTorch;
2. reason about the **shapes** produced by convolutions and pooling;
3. build a compact CNN for Fashion-MNIST;
4. write a correct, device-aware training loop;
5. diagnose a model using accuracy and selected misclassifications.

> **Class method:** before running a cell, predict the result, tensor shape, or behavior. Cells marked **LIVE** are completed together; cells marked **ACTIVITY** are short individual exercises.

> **Instructor version:** includes solutions and teaching notes.


In [ ]:
import time
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)

# Colab: Runtime > Change runtime type > T4 GPU (se disponibile)
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 1. CPU → GPU: what does it really mean? **[LIVE]**

A tensor lives on a *device*. Operations require data and model parameters to be on the same device.

### Predict before running
What do you expect from these three operations?
- create `A` and `B` on the CPU;
- copy them to the GPU with `.to(device)`;
- multiply them with `A @ B`.

For meaningful CUDA benchmarking, two details matter:
- **warm-up**, because the first operation includes initialization overhead;
- `torch.cuda.synchronize()` before stopping the timer, because CUDA kernels execute asynchronously.


In [ ]:
N = 4096  # abbastanza grande da rendere visibile il vantaggio GPU, ma ragionevole su Colab
A_cpu = torch.randn(N, N)
B_cpu = torch.randn(N, N)

# CPU benchmark
t0 = time.perf_counter()
C_cpu = A_cpu @ B_cpu
cpu_time = time.perf_counter() - t0
print(f"CPU matmul: {cpu_time:.3f} s")

if device.type == "cuda":
    # Measure host -> device transfer separately
    t0 = time.perf_counter()
    A_gpu = A_cpu.to(device)
    B_gpu = B_cpu.to(device)
    torch.cuda.synchronize()
    h2d_time = time.perf_counter() - t0

    # warm-up
    _ = A_gpu @ B_gpu
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    C_gpu = A_gpu @ B_gpu
    torch.cuda.synchronize()
    gpu_time = time.perf_counter() - t0

    # ritorno GPU -> CPU (utile per capire che anche l'uscita ha un costo)
    t0 = time.perf_counter()
    C_back = C_gpu.cpu()
    torch.cuda.synchronize()
    d2h_time = time.perf_counter() - t0

    print(f"CPU -> GPU:  {h2d_time:.3f} s")
    print(f"GPU matmul:   {gpu_time:.3f} s")
    print(f"GPU -> CPU:  {d2h_time:.3f} s")
    print(f"speedup matmul (solo compute): {cpu_time / gpu_time:.1f}x")
    print("risultati coerenti:", torch.allclose(C_cpu, C_back, atol=1e-2, rtol=1e-2))
else:
    print("Nessuna GPU CUDA: il benchmark mostra comunque il percorso CPU.")


### ACTIVITY 1 — transfer or computation?

Predict what happens if `N` becomes much smaller, for example `N=256`.

**Question:** is the GPU still necessarily faster *end-to-end*? Why can transfer cost dominate?

> Key message: a GPU is most useful when we perform **a lot of computation per byte transferred** and, during training, keep both model and mini-batches on the GPU instead of repeatedly moving data CPU↔GPU.


## 2. Dataset and mini-batches: inspect before modeling **[LIVE]**

We will use **Fashion-MNIST**: 28×28 grayscale images, 10 classes. It is small enough for live training but non-trivial enough to show why CNNs are useful.

Before running the cell: what shape do you expect for a mini-batch of 128 images?


In [ ]:
transform = transforms.ToTensor()

train_full = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST("data", train=False, download=True, transform=transform)

# Manteniamo una validazione separata.
g = torch.Generator().manual_seed(SEED)
train_data, valid_data = torch.utils.data.random_split(train_full, [55_000, 5_000], generator=g)

# FAST_MODE keeps the live session practical and responsive.
FAST_MODE = True
if FAST_MODE:
    train_data = Subset(train_data, range(12_000))
    valid_data = Subset(valid_data, range(2_000))
    test_live = Subset(test_data, range(2_000))
else:
    test_live = test_data

BATCH_SIZE = 128
pin = device.type == "cuda"
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)
test_loader = DataLoader(test_live, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin)

class_names = train_full.classes
X, y = next(iter(train_loader))
print("X:", X.shape, X.dtype, "range:", (X.min().item(), X.max().item()))
print("y:", y.shape, y.dtype)
print(class_names)

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, label in zip(axes, X[:8], y[:8]):
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(class_names[label])
    ax.axis("off")
plt.tight_layout()


## 3. A convolution before the CNN **[LIVE]**

Before using `nn.Conv2d`, let us make the operation visible. We create a synthetic image with a vertical edge and apply a kernel that measures horizontal intensity differences.

For an input `[B, C, H, W]`, `F.conv2d` expects a kernel with shape `[C_out, C_in, kH, kW]`.


In [ ]:
img = torch.zeros(1, 1, 12, 12)
img[:, :, :, 6:] = 1.0

kernel = torch.tensor([
    [-1., 0., 1.],
    [-1., 0., 1.],
    [-1., 0., 1.],
]).view(1, 1, 3, 3)

response = F.conv2d(img, kernel, padding=1)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(img[0, 0], cmap="gray")
axes[0].set_title("input")
axes[1].imshow(kernel[0, 0], cmap="gray")
axes[1].set_title("kernel")
axes[2].imshow(response[0, 0], cmap="gray")
axes[2].set_title("feature map")
for ax in axes: ax.axis("off")
plt.tight_layout()
print("input:", img.shape, "kernel:", kernel.shape, "output:", response.shape)


## 4. Shape reasoning: padding, stride, pooling **[ACTIVITY]**

For one spatial dimension:

\[
H_{out}=\left\lfloorrac{H+2P-K}{S}ightfloor+1
\]

### ACTIVITY 2 — paper and pencil
For an input of size `28×28`, compute the shapes after:
1. `Conv2d(1, 16, kernel_size=3, padding=1)`;
2. `MaxPool2d(2)`;
3. `Conv2d(16, 32, kernel_size=3, padding=1)`;
4. `MaxPool2d(2)`.

How many features enter the first linear layer?


In [ ]:
probe = torch.randn(4, 1, 28, 28)
blocks = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
)

z = probe
for layer in blocks:
    z = layer(z)
    print(f"{layer.__class__.__name__:12s} -> {tuple(z.shape)}")
print("flatten features/image:", z[0].numel())


## 5. Build the CNN **[LIVE]**

Compact architecture:

`[1,28,28] → Conv(16) → Pool → Conv(32) → Pool → Flatten → 64 → 10`

Questions to ask before completing the code:
- why is the first `in_channels` equal to 1?
- what does `out_channels` represent?
- why do we **not** add Softmax at the output?
- how many parameters does a 3×3 convolution use compared with an equivalent fully connected layer?


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SmallCNN().to(device)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))

X, _ = next(iter(train_loader))
with torch.no_grad():
    logits = model(X[:8].to(device))
print("logits shape:", logits.shape)


## 6. Device-aware training loop **[LIVE]**

With `CrossEntropyLoss`, the model returns **logits**. Each mini-batch follows:

`zero_grad → forward → loss → backward → optimizer.step`.

The important GPU point is to move **each mini-batch** to `device`, while keeping the model there throughout training.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(X)
            loss = criterion(logits, y)
            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * X.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_examples += X.size(0)

    return total_loss / total_examples, total_correct / total_examples

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
EPOCHS = 3 if FAST_MODE else 5
history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, valid_loader, criterion)
    dt = time.perf_counter() - t0
    history.append((train_loss, train_acc, val_loss, val_acc))
    print(
        f"epoch {epoch:02d} | {dt:5.1f}s | "
        f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.3f}"
    )


## 7. Evaluation: more than a single accuracy value **[LIVE]**

Inspect a few misclassified examples. They are often more informative than a single metric.


In [ ]:
test_loss, test_acc = run_epoch(model, test_loader, criterion)
print(f"test loss={test_loss:.4f}, test accuracy={test_acc:.3f}")

model.eval()
X, y = next(iter(test_loader))
with torch.no_grad():
    pred = model(X.to(device)).argmax(dim=1).cpu()

wrong = (pred != y).nonzero(as_tuple=True)[0][:8]
fig, axes = plt.subplots(1, len(wrong), figsize=(14, 2.4))
if len(wrong) == 1:
    axes = [axes]
for ax, i in zip(axes, wrong):
    ax.imshow(X[i, 0], cmap="gray")
    ax.set_title(f"true: {class_names[y[i]]}\npred: {class_names[pred[i]]}")
    ax.axis("off")
plt.tight_layout()


## 8. ACTIVITY 3 — ablation challenge **[ACTIVITY]**

Work individually or in pairs. Choose **one** modification, state a hypothesis first, then test it:

- **A.** replace the second `MaxPool2d(2)` with `Identity` and fix the input size of the `Linear` layer;
- **B.** add `BatchNorm2d` after the convolutional layers;
- **C.** add `Dropout(0.3)` to the classifier;
- **D.** double the channels from `16→32` and `32→64`.

Record three things: **number of parameters**, **training cost**, and **validation accuracy**.
The goal is not to find a single “winner”, but to explain **why** the change affects capacity, computational cost, and generalization.


### Instructor notes for the challenge

If you want to keep the focus on architecture rather than retraining every variant, use a *forward shape check* for some modifications.

Possible discussion points:
- removing pooling preserves more spatial resolution but can make the first `Linear` layer much larger;
- BatchNorm can stabilize or speed up optimization, although the effect may be modest in this small setup;
- Dropout increases regularization but may hurt very short training runs;
- more channels increase model capacity and computational cost, without guaranteeing better results on a small subset.


## 9. Wrap-up

Three ideas to take away:
1. **GPU ≠ magic:** placement, transfers, and synchronization matter;
2. in CNNs, reasoning about **tensor shapes** prevents many errors and clarifies the role of kernels, padding, and pooling;
3. a CNN reuses weights across space: **locality + weight sharing** are the structural reasons it is well suited to images.

### Exit ticket
Write one sentence for each question:
- when can a CPU→GPU transfer cancel out the benefit of GPU computation?
- why does `CrossEntropyLoss` expect logits instead of probabilities already passed through Softmax?
- what is the feature shape immediately before `Flatten` in our network?
